# Lab 2.1 – Datenqualität prüfen

Jede Zelle prüft eine Sache an der Rohdatei. Zellen der Reihe nach mit **Umschalt+Enter** ausführen. Das Notebook ändert die Daten nicht; am Ende steht eine Problemtabelle.

In [1]:
import pandas as pd

ORDNER = "project/examples/m02-data-preparation/l01-data-quality/"   # hier landet die Problemtabelle

df = pd.read_csv("data/nordwind_umsatz.csv")
df.head()

,beleg_nr,datum,jahr,kunde_id,region,produktgruppe,kanal,menge,umsatz,kosten
0,B000001,01.01.2024,2024,K0362,Nord,Schreibwaren,Onlineshop,13,39.89,25.92
1,B000002,01.01.2024,2024,K0281,Nord,Schreibwaren,Onlineshop,37,112.90,80.85
2,B000003,2024-01-01,2024,K0341,Nord,Technik,Außendienst,39,976.83,795.65
3,B000004,2024-01-01,2024,K0397,Nord,Papier,Außendienst,20,184.21,149.64
4,B000005,2024-01-01,2024,K0341,Nord,Ordnung,Außendienst,31,185.20,138.35


## Prüfung 1: Sind alle Pflichtspalten da?

In [2]:
PFLICHT = ["beleg_nr", "datum", "jahr", "kunde_id", "region", "produktgruppe",
           "kanal", "menge", "umsatz", "kosten"]
fehlende_spalten = [s for s in PFLICHT if s not in df.columns]
print("Spalten:", list(df.columns))
print("Fehlende Pflichtspalten:", fehlende_spalten)

Spalten: ['beleg_nr', 'datum', 'jahr', 'kunde_id', 'region', 'produktgruppe', 'kanal', 'menge', 'umsatz', 'kosten']
Fehlende Pflichtspalten: []


## Prüfung 2: Typ und Füllstand jeder Spalte

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33554 entries, 0 to 33553
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   beleg_nr       33554 non-null  object 
 1   datum          33554 non-null  object 
 2   jahr           33554 non-null  int64  
 3   kunde_id       33554 non-null  object 
 4   region         33554 non-null  object 
 5   produktgruppe  33554 non-null  object 
 6   kanal          33554 non-null  object 
 7   menge          33554 non-null  int64  
 8   umsatz         33554 non-null  float64
 9   kosten         32897 non-null  float64
dtypes: float64(2), int64(2), object(6)
memory usage: 2.6+ MB


## Prüfung 3: Fehlende Werte je Spalte

In [4]:
fehlend = df.isna().sum()
fehlend

beleg_nr           0
datum              0
jahr               0
kunde_id           0
region             0
produktgruppe      0
kanal              0
menge              0
umsatz             0
kosten           657
dtype: int64

## Prüfung 4: Dubletten (Zeilen, die genau so schon einmal vorkommen)

In [5]:
dubletten = df.duplicated().sum()
print("Dubletten:", dubletten)

Dubletten: 398


## Prüfung 5: Datumsformate

Der Onlineshop liefert `TT.MM.JJJJ` (mit Punkt), der Außendienst `JJJJ-MM-TT`.

In [6]:
datum_deutsch = df["datum"].str.contains(".", regex=False).sum()
print("Datum mit Punkt (TT.MM.JJJJ):", datum_deutsch)
print("Datum mit Bindestrich (JJJJ-MM-TT):", len(df) - datum_deutsch)

Datum mit Punkt (TT.MM.JJJJ): 7837
Datum mit Bindestrich (JJJJ-MM-TT): 25717


## Prüfung 6: Ausreißer beim Umsatz

`describe()` zeigt Anzahl, Mittelwert, Streuung (`std`), Minimum, Quartile und Maximum.

In [7]:
print(df["umsatz"].describe().round(2))
ausreisser = df[df["umsatz"] > 5000]   # mehr als 5.000 € in einer Zeile
ausreisser

count    33554.00
mean       295.57
std        330.63
min          6.35
25%        131.81
50%        215.27
75%        349.68
max      32800.00
Name: umsatz, dtype: float64


,beleg_nr,datum,jahr,kunde_id,region,produktgruppe,kanal,menge,umsatz,kosten
22956,B022693,2025-11-27,2025,K0007,West,Technik,Außendienst,120,5880.0,5056.8
22957,B022694,2025-11-27,2025,K0007,West,Technik,Außendienst,110,5390.0,4635.4
22959,B022696,2025-11-27,2025,K0007,West,Technik,Außendienst,130,6370.0,5478.2
25450,B025155,2026-02-10,2026,K0207,Ost,Reinigung,Außendienst,44,32800.0,25082.0


## Prüfung 7: Schreibweisen der Regionen – Aufgabe 1

Der Code für diese Zelle kommt von Copilot (Prompt in der Übung, Aufgabe 2). Ergebnis: eine Variable `regionen` mit jeder Schreibweise und ihrer Zeilenzahl.

In [8]:
# >>> AUFGABE 1: Schreibweisen der Spalte region zählen – Prompt siehe Übung, Aufgabe 2
regionen = df["region"].value_counts()   # jede Schreibweise mit ihrer Zeilenzahl
print(regionen)
# <<< ENDE AUFGABE 1

region
Nord           9416
West           8495
Süd            8195
Ost            5746
Region West     255
west            213
Nord            187
nord            165
Region Nord     159
Region Ost      157
Sued            156
süd             151
Region Süd      135
ost             124
Name: count, dtype: int64


In [9]:
# Prüfzelle – nicht ändern, nur ausführen
if "regionen" not in globals():
    print("nicht OK – Variable regionen fehlt: Code aus Aufgabe 1 einfügen und die Zelle darüber ausführen.")
elif not isinstance(regionen, pd.Series) or regionen.sum() != len(df):
    print("nicht OK – regionen soll je Schreibweise die Zeilenzahl enthalten (value_counts), über alle Zeilen.")
elif len(regionen) != df["region"].nunique():
    print(f"nicht OK – {len(regionen)} Schreibweisen, erwartet {df['region'].nunique()}: Leerzeichen und Groß-/Kleinschreibung zählen mit, nichts vorher bereinigen.")
else:
    print(f"OK – {len(regionen)} Schreibweisen für 4 Regionen gefunden.")

OK – 14 Schreibweisen für 4 Regionen gefunden.


## Problemtabelle

Fasst alle Prüfungen zusammen. Die Spalte `Vorschlag` ist die Grundlage für die Bereinigung in Lab 2.2.

In [10]:
anzahl_regionen = len(regionen) if "regionen" in globals() else "fehlt – Aufgabe 1"
probleme = pd.DataFrame([
    ["Dubletten (Zeile doppelt geliefert)", "alle", dubletten, "entfernen"],
    ["Kosten fehlen", "kosten", fehlend["kosten"], "markieren, nicht löschen"],
    ["Datum als TT.MM.JJJJ statt JJJJ-MM-TT", "datum", datum_deutsch, "beide Formate lesen"],
    ["Schreibweisen für 4 Regionen", "region", anzahl_regionen, "vereinheitlichen"],
    ["Umsatz über 5.000 € in einer Zeile", "umsatz", len(ausreisser), "fachlich prüfen"],
], columns=["Problem", "Spalte", "Anzahl", "Vorschlag"])
probleme.to_csv(ORDNER + "probleme.csv", index=False)
print("Gespeichert:", ORDNER + "probleme.csv")
probleme

Gespeichert: project/examples/m02-data-preparation/l01-data-quality/probleme.csv


,Problem,Spalte,Anzahl,Vorschlag
0,Dubletten (Zeile doppelt geliefert),alle,398,entfernen
1,Kosten fehlen,kosten,657,"markieren, nicht löschen"
2,Datum als TT.MM.JJJJ statt JJJJ-MM-TT,datum,7837,beide Formate lesen
3,Schreibweisen für 4 Regionen,region,14,vereinheitlichen
4,Umsatz über 5.000 € in einer Zeile,umsatz,4,fachlich prüfen
